# 02 — Preprocessing

Normalizes and windows the NASA SMAP/MSL dataset for downstream anomaly detection models.

- Fits a `StandardScaler` on the training portion of each entity (first 85% of train set)
- Validates on the last 15% of the train set
- Test windows use stride=1 so that window index `i` corresponds to original timestep `i + WINDOW_SIZE`
- Saves processed arrays and scalers to `data/processed/`

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
import pickle
import json
from pathlib import Path
import os

WINDOW_SIZE = 100
TRAIN_STRIDE = 10
VAL_RATIO = 0.15
DATA_DIR = Path("data")
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(exist_ok=True)

## Loading and Normalization

In [3]:
labels_df = pd.read_csv("labeled_anomalies.csv")

entity_ids = sorted([p.stem for p in (DATA_DIR / "train").glob("*.npy")])

def load_entity(entity_id):
    train_arr = np.load(DATA_DIR / "train" / f"{entity_id}.npy").astype(np.float32)
    test_arr = np.load(DATA_DIR / "test" / f"{entity_id}.npy").astype(np.float32)
    if train_arr.ndim == 1:
        train_arr = train_arr.reshape(-1, 1)
    if test_arr.ndim == 1:
        test_arr = test_arr.reshape(-1, 1)
    return train_arr, test_arr

print(f"Found {len(entity_ids)} entities: {entity_ids[:5]} ...")

Found 82 entities: ['A-1', 'A-2', 'A-3', 'A-4', 'A-5'] ...


In [4]:
normalized = {}

for entity_id in tqdm(entity_ids, desc="Normalizing"):
    try:
        train_arr, test_arr = load_entity(entity_id)

        split_idx = int(len(train_arr) * (1.0 - VAL_RATIO))
        train_part = train_arr[:split_idx]
        val_part = train_arr[split_idx:]

        scaler = StandardScaler()
        scaler.fit(train_part)

        train_norm = scaler.transform(train_part).astype(np.float32)
        val_norm = scaler.transform(val_part).astype(np.float32)
        test_norm = scaler.transform(test_arr).astype(np.float32)

        scaler_path = OUT_DIR / f"{entity_id}_scaler.pkl"
        with open(scaler_path, "wb") as f:
            pickle.dump(scaler, f)

        normalized[entity_id] = (train_norm, val_norm, test_norm)
    except Exception as e:
        print(f"WARNING: skipping {entity_id} — {e}")

print(f"Successfully normalized {len(normalized)} / {len(entity_ids)} entities")

Normalizing:   0%|          | 0/82 [00:00<?, ?it/s]

Successfully normalized 82 / 82 entities


## Windowing

In [5]:
def create_windows(arr, window_size, stride):
    windows = []
    for i in range(0, len(arr) - window_size + 1, stride):
        windows.append(arr[i:i + window_size])
    return np.array(windows, dtype=np.float32)

In [6]:
first_entity = True

for entity_id, (train_norm, val_norm, test_norm) in tqdm(normalized.items(), desc="Windowing"):
    try:
        train_windows = create_windows(train_norm, WINDOW_SIZE, TRAIN_STRIDE)
        val_windows = create_windows(val_norm, WINDOW_SIZE, TRAIN_STRIDE)
        test_windows = create_windows(test_norm, WINDOW_SIZE, stride=1)

        np.save(OUT_DIR / f"{entity_id}_train.npy", train_windows)
        np.save(OUT_DIR / f"{entity_id}_val.npy", val_windows)
        np.save(OUT_DIR / f"{entity_id}_test.npy", test_windows)

        if first_entity:
            print(f"Sanity check — {entity_id}:")
            print(f"  train_windows shape : {train_windows.shape}")
            print(f"  val_windows shape   : {val_windows.shape}")
            print(f"  test_windows shape  : {test_windows.shape}")
            first_entity = False
    except Exception as e:
        print(f"WARNING: windowing failed for {entity_id} — {e}")

Windowing:   0%|          | 0/82 [00:00<?, ?it/s]

Sanity check — A-1:
  train_windows shape : (235, 100, 25)
  val_windows shape   : (34, 100, 25)
  test_windows shape  : (8541, 100, 25)


In [7]:
n_channels = {}
for entity_id, (train_norm, val_norm, test_norm) in normalized.items():
    n_channels[entity_id] = train_norm.shape[1]

metadata = {
    "entities": list(normalized.keys()),
    "n_channels": n_channels,
    "window_size": WINDOW_SIZE,
    "train_stride": TRAIN_STRIDE,
    "val_ratio": VAL_RATIO,
}

with open(OUT_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved metadata.json with {len(metadata['entities'])} entities")

Saved metadata.json with 82 entities


## Verification

In [8]:
sample_id = sorted(normalized.keys())[0]
print(f"Verifying entity: {sample_id}")

train_w = np.load(OUT_DIR / f"{sample_id}_train.npy")
val_w = np.load(OUT_DIR / f"{sample_id}_val.npy")
test_w = np.load(OUT_DIR / f"{sample_id}_test.npy")

print(f"  train_windows : {train_w.shape}")
print(f"  val_windows   : {val_w.shape}")
print(f"  test_windows  : {test_w.shape}")

train_norm, val_norm, _ = normalized[sample_id]
train_last_ts = len(train_norm) - 1
val_first_ts_global = len(train_norm)  # offset within original train array
print(f"  train covers timesteps [0, {len(train_norm)-1}]")
print(f"  val   covers timesteps [{len(train_norm)}, {len(train_norm)+len(val_norm)-1}] (no overlap)")

with open(OUT_DIR / f"{sample_id}_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
print(f"  scaler mean range : [{scaler.mean_.min():.4f}, {scaler.mean_.max():.4f}]")
print(f"  scaler std range  : [{scaler.scale_.min():.4f}, {scaler.scale_.max():.4f}]")
assert not np.any(scaler.scale_ == 0), "Zero std detected!"
assert not np.any(np.abs(scaler.mean_) > 1e6), "Unreasonably large mean!"
print("  Scaler checks passed.")

Verifying entity: A-1
  train_windows : (235, 100, 25)
  val_windows   : (34, 100, 25)
  test_windows  : (8541, 100, 25)
  train covers timesteps [0, 2447]
  val   covers timesteps [2448, 2879] (no overlap)
  scaler mean range : [0.0000, 0.9990]
  scaler std range  : [0.0202, 1.0000]
  Scaler checks passed.


## Dataset Statistics

In [9]:
rows = []
total_train = total_val = total_test = 0

for entity_id in sorted(normalized.keys()):
    try:
        tw = np.load(OUT_DIR / f"{entity_id}_train.npy", mmap_mode="r")
        vw = np.load(OUT_DIR / f"{entity_id}_val.npy", mmap_mode="r")
        tesw = np.load(OUT_DIR / f"{entity_id}_test.npy", mmap_mode="r")
        nc = n_channels[entity_id]
        rows.append({
            "entity": entity_id,
            "n_channels": nc,
            "train_windows": tw.shape[0],
            "val_windows": vw.shape[0],
            "test_windows": tesw.shape[0],
        })
        total_train += tw.shape[0]
        total_val += vw.shape[0]
        total_test += tesw.shape[0]
    except Exception as e:
        print(f"WARNING: could not load stats for {entity_id} — {e}")

stats_df = pd.DataFrame(rows)
print(stats_df.to_string(index=False))
print(f"\nTotal train windows : {total_train}")
print(f"Total val windows   : {total_val}")
print(f"Total test windows  : {total_test}")
print(f"Grand total windows : {total_train + total_val + total_test}")

entity  n_channels  train_windows  val_windows  test_windows
   A-1          25            235           34          8541
   A-2          25            216           30          7815
   A-3          25            223           32          8106
   A-4          25            219           31          7981
   A-5          25             50            1          4594
   A-6          25             48            1          4354
   A-7          25            235           34          8532
   A-8          25             55            2          8276
   A-9          25             55            2          8335
   B-1          25            197           27          7945
   C-1          55            174           23          2165
   C-2          55             55            2          1952
   D-1          25            233           33          8410
  D-11          25            212           30          7332
  D-12          25             17            0          7819
  D-13          25      